In [34]:
from debugpy.launcher import output
from dotenv import load_dotenv
from openai import OpenAI
import os

from openai.types.beta.threads import message

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("API_URL"),
)

In [35]:
from rag_helper import RAGBase
from ingest import load_faq_data, built_index

documents = load_faq_data()
index = built_index(documents)

In [36]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
CONTEXT:
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [37]:
assistant.rag_pipeline("How do I run Olama locally?")

'The provided context does not mention "Olama". It discusses running a course locally instead of using Codespaces, which involves setting up Python, `uv`, Jupyter, Docker, and other tools needed for the module. If you\'re referring to the course, you can run it locally if you are comfortable setting up the required environment and tools.'

In [38]:
assistant.rag_pipeline("How do I run Ollama locally?")

'To run Ollama locally, follow these steps:\n\n1. Install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n   - **macOS**: Download the `.pkg` and install it.\n   - **Windows**: Download the `.msi` and install it.\n   - **Linux**: Run the command `curl -fsSL https://ollama.com/install.sh | sh` in the terminal.\n\n2. Once installed, open a terminal and type `ollama run llama3`. This command will:\n   - Download the LLaMA 3 model (~4GB).\n   - Start the model locally.\n   - Open a chat-like interface where you can type questions.\n\n3. To test the Ollama local server, run the command `curl http://localhost:11434`. You should receive a response similar to `{"models": [...]}`.\n\n4. Install the Python client with `pip install ollama`.\n\nYou can then use the Python client to interact with Ollama locally. A minimal Python example is provided:\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    m

In [58]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [59]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "search in the FAQ database for entries match the given query",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [60]:

instructions="""You are a helpful assistant with access to tools.
    You must call tools using valid JSON ONLY.
    You're task is to Answer the QUESTION based on the CONTEXT from the FAQ database.
    """

In [75]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]


In [77]:
response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    instructions=instructions,
    tools=[search_tool],
)

In [79]:
import json
call = response.output[1]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [80]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [81]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01kw11nsvgeqxsaxk26rc17yf0', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"joining the course"}', call_id='vz00e01vv', name='search', type='function_call', id='vz00e01vv', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'vz00e01vv',
  'output': '[\n  {\n    "id": "9e508f2212",\n    "course": "data-engineering-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: When does the course start?",\n    "answer": "A new cohort runs roughly January\\u2013April every year. For the current cohort\'s exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\\n\\n- Register via the link in the course repo before the cohort starts.\\n- Join the [course Telegra

In [82]:
response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    instructions=instructions,
    tools=[search_tool],
).output_text

In [83]:
response

"Yes, you can still join the course after the start date. According to the course FAQ, you're still eligible to submit homework and the final project, but be aware that there are deadlines for turning in these assignments. It's recommended to get the basic environment ready ahead of time, including setting up a Google Cloud account, installing the Google Cloud SDK, Python 3, Terraform, and Git."